# Module 5 Session 3 (Optional): DeepFilterNet + CI Integration ExperimentsThis notebook covers:1. Experiment design methodology review2. Experiment 1: Modifying ERB band count3. Experiment 2: DeepFilterNet -> ACE pipeline4. Experiment 3: Enhancement evaluation at different SNRs5. Latency analysis for real-time CI applications> "Modifying a model is not random -- every change needs a hypothesis."---

## Section 1: Experiment Design ReviewGood modification experiments require:1. **Hypothesis**: Based on understanding of model and problem2. **Modification**: What to change and how3. **Baseline**: Unmodified model as reference4. **Evaluation**: Metrics to measure effect5. **Analysis**: Do results support the hypothesis?

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport os, sysDFN_DIR = os.path.join('..', 'DeepFilterNet-main', 'DeepFilterNet')ACE_DIR = os.path.join('..', 'module4-deepace', 'ACE')sys.path.insert(0, DFN_DIR)print("Module 5 Session 3: DeepFilterNet + CI Integration")print()# Check available toolstry:    from df.config import config    config.use_defaults()    print("[OK] DeepFilterNet available")    DF_AVAILABLE = Trueexcept:    print("[--] DeepFilterNet not available")    DF_AVAILABLE = Falsetry:    sys.path.insert(0, ACE_DIR)    from ace_strategy import ace_strategy    print("[OK] ACE strategy available")    ACE_AVAILABLE = Trueexcept:    print("[--] ACE strategy not available")    ACE_AVAILABLE = False

## Section 2: Experiment 1 - ERB Band Count Analysis### HypothesisChanging the number of ERB bands affects frequency resolution. Fewer bands = coarser frequency representation but potentially more robust to noise. This is analogous to the CI electrode count tradeoff.### ModificationCompare ERB filterbanks with different numbers of bands: 16, 32, 48.

In [ ]:
# Compare different ERB band countsdef erb_filterbank(n_erbs, sr, n_fft):    n_freqs = n_fft // 2 + 1    freqs = np.linspace(0, sr // 2, n_freqs)    erb_points = 21.4 * np.log10(0.00437 * freqs + 1)    erb_grid = np.linspace(erb_points[0], erb_points[-1], n_erbs + 2)    hz_grid = (10 ** (erb_grid / 21.4) - 1) / 0.00437    fb = np.zeros((n_erbs, n_freqs))    for i in range(n_erbs):        f_left = hz_grid[i]        f_center = hz_grid[i + 1]        f_right = hz_grid[i + 2]        for j in range(n_freqs):            if freqs[j] >= f_left and freqs[j] <= f_center:                fb[i, j] = (freqs[j] - f_left) / (f_center - f_left + 1e-8)            elif freqs[j] > f_center and freqs[j] <= f_right:                fb[i, j] = (f_right - freqs[j]) / (f_right - f_center + 1e-8)    return fbsr = 48000n_fft = 960fig, axes = plt.subplots(1, 3, figsize=(15, 4))for idx, n_erbs in enumerate([16, 32, 48]):    fb = erb_filterbank(n_erbs, sr, n_fft)    axes[idx].imshow(fb, aspect='auto', origin='lower', cmap='hot',                     extent=[0, sr//2, 0, n_erbs])    axes[idx].set_title("ERB: %d bands" % n_erbs)    axes[idx].set_xlabel("Frequency (Hz)")    axes[idx].set_ylabel("Band")plt.tight_layout()plt.show()print("Analysis:")print("  16 bands: Coarser, similar to early CI processors")print("  32 bands: DeepFilterNet default, good balance")print("  48 bands: Finer resolution, closer to normal hearing")print()print("CI connection: Modern CIs have 22 electrodes (between 16 and 32 ERB bands)")

## Section 3: Experiment 2 - Enhance-then-Encode Pipeline### HypothesisApplying speech enhancement before ACE encoding should improve channel selection accuracy, because the enhanced speech has less noise energy contaminating the frequency bands.### Pipeline Design```Noisy speech -> DeepFilterNet -> Enhanced speech -> ACE strategy -> Electrodogram     (Module 5)                                       (Module 4)```Compare with baseline:```Noisy speech -> ACE strategy -> Electrodogram (no enhancement)```

In [ ]:
# Simulate the Enhance-then-Encode pipeline# Generate a test signalsr_ace = 16000  # ACE uses 16kHzduration = 2.0t = np.linspace(0, duration, int(sr_ace * duration), endpoint=False)# Clean speech simulationf0 = 150clean = np.zeros_like(t)for h in range(1, 8):    clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)clean = clean / np.max(np.abs(clean)) * 0.8# Add noisenp.random.seed(42)noise = np.random.randn(len(t)) * 0.5clean_power = np.mean(clean ** 2)noise_power = np.mean(noise ** 2)snr_db = 5noise_scaled = noise * np.sqrt(clean_power / (noise_power * 10 ** (snr_db / 10) + 1e-8))noisy = clean + noise_scalednoisy = noisy / np.max(np.abs(noisy)) * 0.9# Simple spectral subtraction as "enhancement"from scipy.fft import fft, ifftnoisy_fft = fft(noisy)noise_fft = fft(noise_scaled)clean_fft = fft(clean)# Oracle Wiener filter (best case)wiener = (np.abs(clean_fft)**2) / (np.abs(clean_fft)**2 + np.abs(noise_fft)**2 + 1e-8)enhanced = np.real(ifft(wiener * noisy_fft))fig, axes = plt.subplots(3, 1, figsize=(12, 6))axes[0].plot(t[:3200], clean[:3200])axes[0].set_title("Clean Speech")axes[1].plot(t[:3200], noisy[:3200])axes[1].set_title("Noisy (SNR=%d dB)" % snr_db)axes[2].plot(t[:3200], enhanced[:3200])axes[2].set_title("Enhanced (Wiener Filter)")for ax in axes:    ax.set_xlabel("Time (s)")plt.tight_layout()plt.show()print("Enhancement quality:")si_sdr_noisy = 10 * np.log10(np.mean(clean**2) / np.mean((noisy - clean)**2 + 1e-8))si_sdr_enhanced = 10 * np.log10(np.mean(clean**2) / np.mean((enhanced - clean)**2 + 1e-8))print("  Noisy SI-SDR: %.1f dB" % si_sdr_noisy)print("  Enhanced SI-SDR: %.1f dB" % si_sdr_enhanced)

## Section 4: Experiment 3 - SNR Sweep Evaluation### HypothesisEnhancement effectiveness decreases at very low SNRs. There exists an "SNR threshold" below which enhancement provides minimal benefit.

In [ ]:
# SNR sweep evaluationsnr_levels = [-10, -5, 0, 5, 10, 15, 20]results = []for snr in snr_levels:    noise_scaled = noise * np.sqrt(clean_power / (noise_power * 10 ** (snr / 10) + 1e-8))    noisy_snr = clean + noise_scaled    noisy_snr = noisy_snr / (np.max(np.abs(noisy_snr)) + 1e-8) * 0.9    # Wiener enhancement    n_fft_sig = fft(noisy_snr)    c_fft = fft(clean)    n_fft_est = fft(noise_scaled)    wiener_gain = (np.abs(c_fft)**2) / (np.abs(c_fft)**2 + np.abs(n_fft_est)**2 + 1e-8)    enh = np.real(ifft(wiener_gain * n_fft_sig))    # SI-SDR    si_snr_in = 10 * np.log10(np.mean(clean**2) / np.mean((noisy_snr - clean)**2 + 1e-8))    si_snr_out = 10 * np.log10(np.mean(clean**2) / np.mean((enh - clean)**2 + 1e-8))    improvement = si_snr_out - si_snr_in    results.append((snr, si_snr_in, si_snr_out, improvement))    print("SNR=%3ddB: Input SI-SDR=%6.1f, Output SI-SDR=%6.1f, Improvement=%+.1f dB" % (        snr, si_snr_in, si_snr_out, improvement))# Plot resultssnrs = [r[0] for r in results]si_in = [r[1] for r in results]si_out = [r[2] for r in results]improvements = [r[3] for r in results]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))ax1.plot(snrs, si_in, 'o-', label='Input (Noisy)')ax1.plot(snrs, si_out, 's-', label='Output (Enhanced)')ax1.set_xlabel('SNR (dB)')ax1.set_ylabel('SI-SDR (dB)')ax1.set_title('Input vs Output Quality')ax1.legend()ax1.grid(True, alpha=0.3)ax2.bar(range(len(snrs)), improvements, tick_label=[str(s) for s in snrs])ax2.set_xlabel('Input SNR (dB)')ax2.set_ylabel('Improvement (dB)')ax2.set_title('Enhancement Improvement by SNR')ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Section 5: Latency Analysis### Real-time Requirements for CI/Hearing Aids| Application | Max Latency | Algorithm Latency of DF ||------------|-------------|------------------------|| Hearing aids | 5-10 ms | ~5 ms (hop_size/sr = 480/48000) || CI processors | 5-10 ms | ~5 ms || Telephone | ~150 ms | ~5 ms || Video conferencing | ~200 ms | ~5 ms |DeepFilterNet's algorithmic latency = hop_size / sr = 480 / 48000 = 10 msBut with 50% overlap, effective latency is about 5 ms.**Challenges for CI deployment:**1. **Compute budget**: CI processors have very limited compute (~10x less than smartphone)2. **Power consumption**: Must run on battery for 12+ hours3. **Model size**: DeepFilterNet3 has ~1.5M parameters4. **Fixed-point arithmetic**: CI hardware may not support floating-point**Research direction**: Model compression, quantization, pruning for CI deployment.

## Section 6: Discussion - From Speech Enhancement to CI Enhancement### The GapSpeech enhancement (SE) optimizes for general listeners. CI users have unique needs:| General SE | CI-specific SE ||-----------|---------------|| Optimize PESQ | Optimize STOI (intelligibility) || Full-band enhancement | Focus on CI-relevant frequencies || Any noise type | CI-typical environments (restaurants, streets) || Single output | Must feed into CI encoder (ACE) || Latency: nice to have | Latency: mandatory <10ms |### Bridging the GapPotential research directions:1. **CI-aware loss function**: Include ACE channel selection quality in training loss2. **Joint optimization**: Train SE + CI encoder end-to-end3. **Per-electrode enhancement**: Apply different enhancement to each CI channel4. **Noise type adaptation**: Focus on noise types most problematic for CI users5. **Subjective evaluation with CI users**: The gold standard---## SummaryThis session covered:1. ERB band count analysis and its CI connection2. Enhance-then-Encode pipeline design3. SNR sweep evaluation methodology4. Latency requirements for real-time CI applications5. The gap between general SE and CI-specific SE**Homework:**- Basic: Run DeepFilterNet inference on 3 different SNR samples, compute metrics- Advanced: Build a complete Enhance -> ACE pipeline and compare channel selection with/without enhancement